In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

L = d *f

dL/dd =? f

(f(x+h) -f(x))/h
((d+h)*f - d*f)/h
(h*f)/h
f

In [2]:
class Value:
    def __init__(self, data, _children=(), _op='', label =''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')

        def _backward():
          self.grad += 1.0 * out.grad
          other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
      assert isinstance(other,(int,float)), "only supporting int/float powers for now"
      out = Value(self.data**other, (self,), f'**{other}')

      def _backword():
        self.grad += other * (self.data ** (other -1)) * out.grad
      out._backward = _backword


      return out


    def __rmul__(self, other):
        return self * other

    def __truediv__(self,other):
      return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)


    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self,other), '*')

        def _backward():
          self.grad += (other.data * out.grad)
          other.grad += (self.data * out.grad)
        out._backward = _backward

        return out

    def tanh(self):
      x = self.data
      t = (math.exp(2*x)-1 )/(math.exp(2*x)+1)
      out = Value(t, (self, ), 'tanh')

      def _backward():
        self.grad += (1- t**2) * out.grad
      out._backward = _backward
      return out

    def exp(self):
      x = self.data
      out = Value(math.exp(x), (self, ),'exp')

      def _backword():
        self.grad += out.data * out.grad

      out._backward = _backword

      return out

    def backward(self):

      topo = []
      visited = set()
      def build_topo(v):
        if v not in visited:
          visited.add(v)
          for child in v._prev:
            build_topo(child)
          topo.append(v)
      build_topo(self)
      self.grad = 1.0
      for node in reversed(topo):
        node._backward()



In [3]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format ='svg', graph_attr={'rankdir':'LR'})

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name = uid, label = "{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape="record")
        if n._op:
            dot.node(name = uid + n._op, label = n._op)
            dot.edge(uid + n._op, uid)


    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [4]:
import random

In [5]:
class Neuron:

  def __init__(self,nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))

  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
    out = act.tanh()
    return out


  def parameters(self):
    return self.w + [self.b]

class Layer:

  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self,x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]
    # params = []
    # for neuron in self.neurons:
    #   ps = neuron.parameter()
    #   params.extend(ps)
    # return params


class MLP:
  def __init__(self,nin,nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

  def __call__(self,x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]



In [6]:
x = [2.0,3.0, -1.0]
n = MLP(3, [4,4,1])
n(x)

Value(data=-0.06248077326209538)

In [7]:
xs = [
    [2.0,3.0,-1.0],
    [3.0,-1.0,0.5],
    [0.5,1.0,1.0],
    [1.0,1.0,-1.0],
]
ys = [1.0,-1.0,-1.0,1.0]


In [10]:
for k in range(20):
  #forward pass
  ypred = [n(x) for x in xs]
  loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0.0))

  #backward pass
  for p in n.parameters():
    p.grad =0.0
  loss.backward()

  #update
  for p in n.parameters():
    p.data += -0.05 * p.grad

  print(k,loss.data)


0 0.04853015983945411
1 0.04531123743298627
2 0.042461226942359466
3 0.039922237148216966
4 0.037647653344762916
5 0.03559955508183213
6 0.03374680555244422
7 0.032063618833802626
8 0.030528472163872684
9 0.029123270758275362
10 0.027832699797202574
11 0.02664371674915126
12 0.025545150052863706
13 0.024527379214607476
14 0.023582077809980777
15 0.022702005512248463
16 0.021880838642987344
17 0.02111303122315266
18 0.020393700346839568
19 0.01971853108256934
20 0.01908369715232929
21 0.018485794435454263
22 0.017921784956499105
23 0.017388949489768834
24 0.01688484728201913
25 0.016407281684023313
26 0.01595427070982089
27 0.015524021723480236
28 0.01511490959764094
29 0.014725457803967107
30 0.01435432198905944
31 0.014000275665059465
32 0.013662197705783493
33 0.0133390613895889
34 0.013029924771529627
35 0.012733922201448445
36 0.012450256832870859
37 0.012178193991005486
38 0.011917055287701223
39 0.01166621338756302
40 0.011425087343155683
41 0.011193138428787502
42 0.010969866412

In [11]:
ypred

[Value(data=0.9562973112642628),
 Value(data=-0.961083835289829),
 Value(data=-0.9437949032505045),
 Value(data=0.9449439002653425)]